# The Transformer Architecture

**Companion lesson:** https://ml-viz.vercel.app/courses/transformers/03-transformer-architecture

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A full Transformer block in numpy

Assemble the pieces: multi-head self-attention + a position-wise feed-forward net, each wrapped in a **residual** connection and **LayerNorm** (pre-LN style).

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x)
    return e/e.sum(axis=axis, keepdims=True)

def layernorm(x, eps=1e-5):
    mu = x.mean(-1, keepdims=True); var = x.var(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

def mha(X, Wq, Wk, Wv, Wo, h, mask=None):
    T, d = X.shape; dk = d//h
    Q=(X@Wq).reshape(T,h,dk).transpose(1,0,2); K=(X@Wk).reshape(T,h,dk).transpose(1,0,2)
    V=(X@Wv).reshape(T,h,dk).transpose(1,0,2)
    s=Q@K.transpose(0,2,1)/np.sqrt(dk)
    if mask is not None: s=np.where(mask, s, -1e9)
    ctx=(softmax(s)@V).transpose(1,0,2).reshape(T,d)
    return ctx@Wo

def relu(x): return np.maximum(0, x)

class TransformerBlock:
    def __init__(self, d, h, d_ff, seed=0):
        rng=np.random.RandomState(seed); self.h=h
        self.Wq,self.Wk,self.Wv,self.Wo = (rng.randn(d,d)*0.1 for _ in range(4))
        self.W1=rng.randn(d,d_ff)*0.1; self.b1=np.zeros(d_ff)
        self.W2=rng.randn(d_ff,d)*0.1; self.b2=np.zeros(d)
    def __call__(self, X, mask=None):
        X = X + mha(layernorm(X), self.Wq,self.Wk,self.Wv,self.Wo, self.h, mask)  # residual
        ff = relu(layernorm(X)@self.W1 + self.b1)@self.W2 + self.b2
        return X + ff                                                            # residual

d_model, n_heads, d_ff, T = 32, 4, 128, 7
X = np.random.RandomState(1).randn(T, d_model)
block = TransformerBlock(d_model, n_heads, d_ff)
print('block output:', block(X).shape, '(same shape in, same shape out)')

## Stacking blocks = a deep Transformer

Real models stack dozens of identical blocks. Residuals keep activations stable as depth grows — we track the activation norm through the stack.

In [ ]:
blocks = [TransformerBlock(d_model, n_heads, d_ff, seed=i) for i in range(12)]
norms = [np.linalg.norm(X)]
h = X
for blk in blocks:
    h = blk(h); norms.append(np.linalg.norm(h))
plt.plot(norms, 'o-', color='#6366f1'); plt.xlabel('block depth'); plt.ylabel('||activations||')
plt.title('Residual connections keep a 12-block stack stable'); plt.show()

## Decoder-only forward pass (GPT-style)

Add token embeddings + positional encoding, run causal blocks, project to vocab logits, softmax → next-token distribution. Here, untrained, just to show the data flow and shapes.

In [ ]:
vocab, d_model, T = 50, 32, 7
rng = np.random.RandomState(2)
tok_emb = rng.randn(vocab, d_model)*0.1
def pos_enc(T, d):
    p=np.arange(T)[:,None]; i=np.arange(d)[None,:]; a=p/np.power(10000,(2*(i//2))/d)
    pe=np.zeros((T,d)); pe[:,0::2]=np.sin(a[:,0::2]); pe[:,1::2]=np.cos(a[:,1::2]); return pe

tokens = rng.randint(0, vocab, size=T)
Xh = tok_emb[tokens] + pos_enc(T, d_model)
mask = np.tril(np.ones((T, T))).astype(bool)
for blk in blocks:
    Xh = blk(Xh, mask=mask)
W_out = rng.randn(d_model, vocab)*0.1
logits = layernorm(Xh) @ W_out
probs = softmax(logits)
print('next-token distribution shape:', probs.shape)
print('predicted next token after position', T-1, '->', int(probs[-1].argmax()))

## Key takeaways

- A block alternates **attention** (mix tokens) and a **feed-forward** net (per-token compute).
- **Residuals + LayerNorm** are what make deep stacks trainable — activation norm stays bounded.
- A causal mask turns the encoder block into a GPT-style **decoder**.
- The full forward pass is: embed + position → N blocks → project → softmax over the vocab.